In [1]:
import requests
import brotli
import json
import re
import time
import datetime

In [2]:
url = 'https://api.skinport.com/v1/items'
response = requests.get(url, headers={"Accept-Encoding": "br"})
print("status:", response.status_code)


status: 200


In [7]:
if response.status_code == 200:
    try:
        data = response.json()
    except json.JSONDecodeError:
        raw = brotli.decompress(response.content)
        data = json.loads(raw)
    ts = time.time()
    time_stamp = datetime.datetime.fromtimestamp(ts).strftime('%Y-%m-%d %H:%M:%S')
    print(len(data), "items")
    print(data[0])
else:
    print(response.text[:300])

25441 items
{'market_hash_name': 'Patch | Phoenix', 'version': None, 'currency': 'EUR', 'suggested_price': 0.33, 'item_page': 'https://skinport.com/item/patch-phoenix', 'market_page': 'https://skinport.com/market/patch?item=Phoenix', 'min_price': 0.27, 'max_price': 2.3, 'mean_price': 0.84, 'median_price': 1.1, 'quantity': 174, 'created_at': 1583227179, 'updated_at': 1788473414}


In [8]:
data = response.json()

In [9]:
data[:30]

[{'market_hash_name': 'Patch | Phoenix',
  'version': None,
  'currency': 'EUR',
  'suggested_price': 0.33,
  'item_page': 'https://skinport.com/item/patch-phoenix',
  'market_page': 'https://skinport.com/market/patch?item=Phoenix',
  'min_price': 0.27,
  'max_price': 2.3,
  'mean_price': 0.84,
  'median_price': 1.1,
  'quantity': 174,
  'created_at': 1583227179,
  'updated_at': 1788473414},
 {'market_hash_name': 'Patch | Rage',
  'version': None,
  'currency': 'EUR',
  'suggested_price': 4.88,
  'item_page': 'https://skinport.com/item/patch-rage',
  'market_page': 'https://skinport.com/market/patch?item=Rage',
  'min_price': 4.51,
  'max_price': 10.09,
  'mean_price': 6.74,
  'median_price': 5.24,
  'quantity': 22,
  'created_at': 1583227179,
  'updated_at': 1788473414},
 {'market_hash_name': 'AWP | Acheron (Factory New)',
  'version': None,
  'currency': 'EUR',
  'suggested_price': 15.49,
  'item_page': 'https://skinport.com/item/awp-acheron-factory-new',
  'market_page': 'https://sk

In [10]:
#extract kv pairs i am interested in
extract_kv = ['market_hash_name', 'min_price', 'quantity']
extracted_list = []

for i in data:
    temp = {}
    for k in extract_kv:
        if k in i:
            temp[k] = i[k]
    extracted_list.append(temp)

In [11]:
 #I need a separate field in the dict for the wear/stattrak(★)/souvenir/sticker(Gold,Holo,Foil,Glitter) of the item.


#Ok so I need to add fields, extract it from the market_hash_name, maybe add external general item dataset to crosscheck:



In [12]:
for item in extracted_list:
    print(item['market_hash_name'])



Patch | Phoenix
Patch | Rage
AWP | Acheron (Factory New)
AWP | Acheron (Field-Tested)
AWP | Acheron (Minimal Wear)
AWP | Acheron (Well-Worn)
AWP | Arsenic Spill (Battle-Scarred)
AWP | Arsenic Spill (Factory New)
AWP | Arsenic Spill (Field-Tested)
AWP | Arsenic Spill (Minimal Wear)
AWP | Arsenic Spill (Well-Worn)
AWP | Asiimov (Battle-Scarred)
AWP | Asiimov (Field-Tested)
AWP | Asiimov (Well-Worn)
AWP | Atheris (Battle-Scarred)
AWP | Atheris (Factory New)
AWP | Atheris (Field-Tested)
AWP | Atheris (Minimal Wear)
10 Year Birthday Sticker Capsule
1st Lieutenant Farlow | SWAT
Aces High Pin
AUG | Carved Jade (Minimal Wear)
AUG | Chameleon (Battle-Scarred)
AUG | Chameleon (Factory New)
AUG | Chameleon (Field-Tested)
AUG | Chameleon (Minimal Wear)
AUG | Chameleon (Well-Worn)
AUG | Colony (Battle-Scarred)
AUG | Colony (Field-Tested)
AUG | Colony (Minimal Wear)
AUG | Colony (Well-Worn)
AUG | Creep (Field-Tested)
AUG | Creep (Minimal Wear)
AUG | Creep (Well-Worn)
AUG | Daedalus (Battle-Scarred)


In [13]:
dupey = extracted_list
len(dupey)

25441

In [14]:
"""Item:
    raw_name: str            # the original market_hash_name
    item_name: str            # the base name with wear/quality stripped, e.g. "Talon Knife"
    item_type: str            # "weapon"  "knife"  "gloves"  "sticker"  "other"
    quality_tag: str | None   # "StatTrak/StatTrak™/★"  "Souvenir" | None  — mutually exclusive, weapons/knives only
    wear: str | None          # "Factory New"  ...  "Battle-Scarred" None for item_types that don't have wear
    sticker_finish: str | None  # "Foil"  "Holo"  "Gold"  "Glitter"  "Embroidered"  None — only populated when item_type == "sticker"
"""



for item in dupey:
    name = item['market_hash_name']
    item['ts'] = time_stamp
    item['venue'] = 'Skinport'
    if 'StatTrak' in name:
        item['quality_tag'] = 'StatTrak'
    elif 'Souvenir' in name:
        item['quality_tag'] = 'Souvenir'
    else:
        item['quality_tag'] = None


    if 'Gloves' in name:
        item['item_type'] = 'Gloves'
    elif 'Graffiti' in name:
        item['item_type'] = 'Graffiti'
    elif 'Charm' in name:
        item['item_type'] = 'Charm'
    elif '★' in name or 'Knife' in name and 'Gloves' not in name:
        item['item_type'] = 'Knife'
    elif 'Sticker Capsule' in name and not 'Sticker Slab' in name:
        item['item_type'] = 'Sticker Capsule'
    elif 'Sticker Slab' in name:
        item['item_type'] = 'Sticker Slab'
    elif 'Sticker' in name and not 'Sticker Slab' in name:
        item['item_type'] = 'Sticker'
    elif 'Patch |' in name:
        item['item_type'] = 'Patch'
    elif 'Music Kit' in name:
        item['item_type'] = 'Music Kit'
    else:
        item['item_type'] = 'Weapon'

    if 'Factory New' in name:
        item['wear'] = 'Factory New'
    elif 'Minimal Wear' in name:
        item['wear'] = 'Minimal Wear'
    elif 'Field-Tested' in name:
        item['wear'] = 'Field-Tested'
    elif 'Well-Worn' in name:
        item['wear'] = 'Well-Worn'
    elif 'Battle-Scarred' in name:
        item['wear'] = 'Battle-Scarred'
    else:
        item['wear'] = None


    if item['item_type'] == 'Sticker':
        if 'Foil' in name:
            item['sticker_finish'] = 'Foil'
        elif 'Holo' in name:
            item['sticker_finish'] = 'Holo'
        elif 'Gold' in name:
                item['sticker_finish'] = 'Gold'
        elif 'Glitter' in name:
            item['sticker_finish'] = 'Glitter'
        elif 'Embroidered' in name:
            item['sticker_finish'] = 'Embroidered'
        else:
            item['sticker_finish'] = None


In [15]:
len(dupey)

25441

In [16]:
dupey_clean = []
for item in dupey:
    if item['item_type'] in ('Weapon', 'Knife', 'Gloves') and item['wear'] is not None:
        dupey_clean.append(item)
    elif item['item_type'] in ('Sticker', 'Sticker Capsule', 'Sticker Slab', 'Patch', 'Graffiti', 'Charm', 'Music Kit'):
        dupey_clean.append(item)
    else:
        continue

len(dupey_clean)

24852

In [17]:
for item in dupey_clean:
    word = item['market_hash_name']
    clean_text = word.replace("★ StatTrak™ ", "")
    clean_text = clean_text.replace("StatTrak™ ", "")
    clean_text = clean_text.replace("Souvenir ", "")
    clean_text = clean_text.replace(" Sticker Capsule", "")
    clean_text = clean_text.replace("★ ", "")
    clean_text = clean_text.replace("Sticker | ", "")
    clean_text = clean_text.replace("Music Kit | ", "")
    clean_text = re.sub(r"\s*\([^)]*\)", "", clean_text)
    item['market_hash_name'] = clean_text

len(dupey_clean)


24852

In [18]:
dupey

[{'market_hash_name': 'Patch | Phoenix',
  'min_price': 0.27,
  'quantity': 174,
  'ts': '2026-09-04 00:15:10',
  'venue': 'Skinport',
  'quality_tag': None,
  'item_type': 'Patch',
  'wear': None},
 {'market_hash_name': 'Patch | Rage',
  'min_price': 4.51,
  'quantity': 22,
  'ts': '2026-09-04 00:15:10',
  'venue': 'Skinport',
  'quality_tag': None,
  'item_type': 'Patch',
  'wear': None},
 {'market_hash_name': 'AWP | Acheron',
  'min_price': 11.53,
  'quantity': 95,
  'ts': '2026-09-04 00:15:10',
  'venue': 'Skinport',
  'quality_tag': None,
  'item_type': 'Weapon',
  'wear': 'Factory New'},
 {'market_hash_name': 'AWP | Acheron',
  'min_price': 0.73,
  'quantity': 119,
  'ts': '2026-09-04 00:15:10',
  'venue': 'Skinport',
  'quality_tag': None,
  'item_type': 'Weapon',
  'wear': 'Field-Tested'},
 {'market_hash_name': 'AWP | Acheron',
  'min_price': 1.95,
  'quantity': 83,
  'ts': '2026-09-04 00:15:10',
  'venue': 'Skinport',
  'quality_tag': None,
  'item_type': 'Weapon',
  'wear': '